# Baseline RAG - Colab Runner
Thin runner notebook. Core logic stays in the Python modules.

## Clone private repo
Use a GitHub Personal Access Token with read access to this repo.

In [ ]:
import getpass
import os
import subprocess

os.chdir('/content')

token = getpass.getpass('GitHub token (repo read access): ')
result = subprocess.run(
    ['git', 'clone', f'https://{token}@github.com/zahiTouqan/latent-rag.git'],
    capture_output=True, text=True,
)
if result.returncode != 0:
    # Strip token from error output before printing
    print('STDOUT:', result.stdout.replace(token, '***'))
    print('STDERR:', result.stderr.replace(token, '***'))
    raise SystemExit('Clone failed')
else:
    print('Clone successful')

In [ ]:
%cd /content/latent-rag
!python -m pip install -U pip
!pip install -r requirements.txt
# Pin datasets<3.0 to retain support for script-based loaders (e.g. facebook/kilt_wikipedia)
!pip install 'datasets>=2.14,<3'

## Build a KILT Wikipedia passage corpus
Streams `facebook/kilt_wikipedia` so you do not need to download the full corpus in Colab.

In [ ]:
import json

from datasets import load_dataset

CORPUS_OUT = '/content/wiki_passages.jsonl'
MAX_DOCS = 5000
PARAGRAPHS_PER_PASSAGE = 3

def normalize_title(value):
    if value is None:
        return ''
    return str(value).replace('_', ' ').strip()

print('Streaming KILT Wikipedia ...')
wiki_ds = load_dataset('facebook/kilt_wikipedia', split='full', streaming=True)

indexed_doc_ids = set()
doc_count = 0
passage_count = 0
with open(CORPUS_OUT, 'w', encoding='utf-8') as handle:
    for row in wiki_ds:
        if doc_count >= MAX_DOCS:
            break

        doc_id = normalize_title(row.get('wikipedia_title'))
        if not doc_id:
            continue

        paragraphs = [
            paragraph.strip()
            for paragraph in row.get('text', {}).get('paragraph', [])
            if isinstance(paragraph, str) and paragraph.strip()
        ]
        if not paragraphs:
            continue

        wikipedia_id = str(row.get('wikipedia_id', '')).strip() or doc_id
        for start in range(0, len(paragraphs), PARAGRAPHS_PER_PASSAGE):
            block = paragraphs[start:start + PARAGRAPHS_PER_PASSAGE]
            text = '\n'.join(block).strip()
            if not text:
                continue
            passage_id = f'{wikipedia_id}:{start // PARAGRAPHS_PER_PASSAGE}'
            handle.write(json.dumps({'id': passage_id, 'doc_id': doc_id, 'text': text}) + '\n')
            passage_count += 1

        indexed_doc_ids.add(doc_id)
        doc_count += 1
        if doc_count % 1000 == 0:
            print(f'  processed {doc_count} docs -> {passage_count} passages')

print(f'Done. Wrote {passage_count} passages from {doc_count} docs to {CORPUS_OUT}')

## Build Natural Questions eval file
Loads `google-research-datasets/natural_questions` and converts it into the repo's eval format.

In [ ]:
import itertools

EVAL_OUT = '/content/natural_questions_eval.jsonl'
TARGET_EVAL = 200
FILTER_EVAL_TO_INDEXED_DOCS = False

def extract_short_answers(example):
    answers = []
    ann = example.get('annotations') or {}

    # short_answers: list of annotator dicts, each with 'text' -> list[str]
    for sa in ann.get('short_answers', []):
        for text in sa.get('text', []):
            if isinstance(text, str) and text.strip():
                answers.append(text.strip())

    # yes_no_answer: list of ints (-1 = none, 0 = NO, 1 = YES)
    for yn in ann.get('yes_no_answer', []):
        if yn == 1:
            answers.append('yes')
        elif yn == 0:
            answers.append('no')

    return list(dict.fromkeys(answers))

print(f'Streaming Natural Questions validation (target {TARGET_EVAL} eval rows) ...')
nq_ds = load_dataset('google-research-datasets/natural_questions', split='validation', streaming=True)

eval_count = 0
scanned = 0
skip_reasons = {'no_question': 0, 'no_answers': 0, 'no_doc_id': 0, 'not_indexed': 0}
with open(EVAL_OUT, 'w', encoding='utf-8') as handle:
    for row in nq_ds:
        scanned += 1
        question = str(row.get('question', {}).get('text', '')).strip()
        answers = extract_short_answers(row)
        doc_id = normalize_title(row.get('document', {}).get('title'))

        if not question:
            skip_reasons['no_question'] += 1
            continue
        if not answers:
            skip_reasons['no_answers'] += 1
            continue
        if not doc_id:
            skip_reasons['no_doc_id'] += 1
            continue
        if FILTER_EVAL_TO_INDEXED_DOCS and doc_id not in indexed_doc_ids:
            skip_reasons['not_indexed'] += 1
            continue

        handle.write(json.dumps({
            'question': question,
            'answer': answers,
            'relevant_ids': [doc_id],
        }) + '\n')
        eval_count += 1
        if eval_count >= TARGET_EVAL:
            break

print(f'Wrote {eval_count} eval rows to {EVAL_OUT} (scanned {scanned})')
skipped = sum(skip_reasons.values())
if skipped:
    print(f'Skipped {skipped} rows: {skip_reasons}')

## Build the index
This creates `index.faiss`, `metadata.jsonl`, and `config.json`.

In [ ]:
INDEX_DIR = '/content/index'
!python build_index.py --corpus_path {CORPUS_OUT} --index_dir {INDEX_DIR}


## Evaluate
Natural Questions is aligned here by document title, so recall is computed against `source_doc_id`.

In [ ]:
!python evaluate.py --eval_path {EVAL_OUT} --index_dir {INDEX_DIR} --retrieval_id_field source_doc_id
